In [ ]:
import numpy as np
from xgboost import XGBClassifier
import pandas as pd
from category_encoders import MEstimateEncoder
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.feature_selection import mutual_info_regression
from sklearn.model_selection import KFold, cross_val_score, RandomizedSearchCV
from xgboost import XGBRegressor

o_encoder = OrdinalEncoder()
oh_encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

In [ ]:
def make_mi_scores(X, y):
    X = X.copy()
    for colname in X.select_dtypes(["object", "category"]):
        X[colname], _ = X[colname].factorize()
    discrete = [X[c].nunique() <= 15 for c in X.columns]
    scores = mutual_info_regression(X, y, discrete_features=discrete, random_state=0)
    return pd.Series(scores, name="MI Scores", index=X.columns).sort_values(ascending=False)

In [ ]:
X_train = pd.read_csv('train.csv', na_values=['NA'], keep_default_na=False)
X_test = pd.read_csv('test.csv', na_values=['NA'], keep_default_na=False)


In [ ]:
y_train = X_train['SalePrice']
X_train = X_train.drop('SalePrice', axis='columns')

# Drop 2 known bad rows (huge living area, top quality, sold far too cheap) up front, before
# any encoder or statistic is fit on the training data. The test set is never row-filtered.
outlier_mask = (X_train["GrLivArea"] > 4000) & (y_train < 300000)
X_train = X_train.loc[~outlier_mask]
y_train = y_train.loc[~outlier_mask]

# Competition metric is RMSE of log(SalePrice); log1p also de-skews the target (1.88 -> 0.12).
# Predictions must be passed back through np.expm1 before submitting.
y_train = np.log1p(y_train)


In [ ]:
FILL_NONE = [
    "PoolQC", "MiscFeature", "Alley", "Fence", "FireplaceQu",
    "GarageType", "GarageFinish", "GarageQual", "GarageCond",
    "BsmtQual", "BsmtCond", "BsmtExposure", "BsmtFinType1", "BsmtFinType2",
    "MasVnrType",
]   # Fill these with 'None' since they don't exist in the home and are categorical

X_train[FILL_NONE] = X_train[FILL_NONE].fillna('None')
X_test[FILL_NONE] = X_test[FILL_NONE].fillna('None')

In [ ]:
FILL_ZERO = [
    "MasVnrArea", "GarageYrBlt",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "BsmtFullBath", "BsmtHalfBath",
    "GarageCars", "GarageArea",
]   # Fill with 0 since they don't exist and are numerical

X_train[FILL_ZERO] = X_train[FILL_ZERO].fillna(0)
X_test[FILL_ZERO] = X_test[FILL_ZERO].fillna(0)

In [ ]:

PORCH_COLS = ["OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch"]

for df in (X_train, X_test):
    df["TotalSF"] = df["TotalBsmtSF"] + df["1stFlrSF"] + df["2ndFlrSF"]          # total finished + unfinished area
    df["TotalBath"] = (df["FullBath"] + 0.5 * df["HalfBath"]
                       + df["BsmtFullBath"] + 0.5 * df["BsmtHalfBath"])          # half-baths count half
    df["TotalPorchSF"] = df[PORCH_COLS].sum(axis=1) + df["WoodDeckSF"]           # all outdoor deck/porch area
    df["OverallScore"] = df["OverallQual"] * df["OverallCond"]

In [ ]:
PRESENCE_FLAGS = {
    "HasGarage":   "GarageArea",   # 1 if > 0
    "HasBsmt":     "TotalBsmtSF",
    "HasFireplace": "Fireplaces",
    "HasPool":     "PoolArea",
    "Has2ndFloor": "2ndFlrSF",
    "HasPorch":    None,            # OpenPorchSF+EnclosedPorch+3SsnPorch+ScreenPorch > 0
}

for flag, src in PRESENCE_FLAGS.items():
    for df in (X_train, X_test):
        if src is None:                      # HasPorch: any porch area > 0
            df[flag] = (df[PORCH_COLS].sum(axis=1) > 0).astype(int)
        else:
            df[flag] = (df[src] > 0).astype(int)   # per-row: 1 if present, else 0

In [ ]:
GROUP_IMPUTE = {"LotFrontage": "Neighborhood"}   # median within Neighborhood, global-median fallback

for target, group_col in GROUP_IMPUTE.items():
    grp_median = X_train.groupby(group_col)[target].median()
    global_median = X_train[target].median()   # fallback for a group absent from train
    for df in (X_train, X_test):
        df[target] = df[target].fillna(df[group_col].map(grp_median)).fillna(global_median)


In [ ]:
MODE_IMPUTE = [   # stray single NaNs (mostly test only)
    "MSZoning", "Functional", "Exterior1st", "Exterior2nd",
    "KitchenQual", "SaleType", "Electrical",
]

for i in MODE_IMPUTE:
    mode = X_train[i].mode()[0]
    for df in (X_train, X_test):
        df[i] = df[i].fillna(mode)

In [ ]:
QUAL = {"None": 0, "Po": 1, "Fa": 2, "TA": 3, "Gd": 4, "Ex": 5}
ORDINAL_MAPS = {
    "ExterQual":   QUAL, "ExterCond":   QUAL, "HeatingQC": QUAL, "KitchenQual": QUAL,
    "FireplaceQu": QUAL, "GarageQual":  QUAL, "GarageCond": QUAL,
    "BsmtQual":    QUAL, "BsmtCond":    QUAL, "PoolQC":     QUAL,
    "BsmtExposure": {"None": 0, "No": 1, "Mn": 2, "Av": 3, "Gd": 4},
    "BsmtFinType1": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "BsmtFinType2": {"None": 0, "Unf": 1, "LwQ": 2, "Rec": 3, "BLQ": 4, "ALQ": 5, "GLQ": 6},
    "GarageFinish": {"None": 0, "Unf": 1, "RFn": 2, "Fin": 3},
    "Functional":   {"Sal": 0, "Sev": 1, "Maj2": 2, "Maj1": 3, "Mod": 4, "Min2": 5, "Min1": 6, "Typ": 7},
    "LotShape":     {"IR3": 0, "IR2": 1, "IR1": 2, "Reg": 3},
    "LandSlope":    {"Sev": 0, "Mod": 1, "Gtl": 2},
    "PavedDrive":   {"N": 0, "P": 1, "Y": 2},
    "CentralAir":   {"N": 0, "Y": 1},
}
ORDINAL_COLS = list(ORDINAL_MAPS)

for target, rank in ORDINAL_MAPS.items():
    for df in (X_train, X_test):
        df[target] = df[target].map(rank)

In [ ]:
CAST_TO_STR = ["MSSubClass"]

for df in (X_train, X_test):
    df[CAST_TO_STR] = df[CAST_TO_STR].astype(str)

In [ ]:
NOMINAL_COLS = [
    "MSSubClass",   # after cast to str
    "MSZoning", "Street", "Alley", "LandContour", "LotConfig", "Neighborhood",
    "Condition1", "Condition2", "BldgType", "HouseStyle", "RoofStyle", "RoofMatl",
    "Exterior1st", "Exterior2nd", "MasVnrType", "Foundation", "Heating",
    "Electrical", "GarageType", "Fence", "MiscFeature", "SaleType", "SaleCondition",
]

oh_encoder.fit(X_train[NOMINAL_COLS])

names = oh_encoder.get_feature_names_out(NOMINAL_COLS)                    # e.g. "Neighborhood_NAmes", ...

# transform each frame, keep the original row index so concat lines up
train_oh = pd.DataFrame(oh_encoder.transform(X_train[NOMINAL_COLS]), columns=names, index=X_train.index)
test_oh  = pd.DataFrame(oh_encoder.transform(X_test[NOMINAL_COLS]),  columns=names, index=X_test.index)

# drop the raw text columns, attach the 0/1 columns
X_train = pd.concat([X_train.drop(columns=NOMINAL_COLS), train_oh], axis=1)
X_test  = pd.concat([X_test.drop(columns=NOMINAL_COLS),  test_oh],  axis=1)

In [ ]:
YEAR_COLS = ["YearBuilt", "YearRemodAdd", "GarageYrBlt", "YrSold"]  # candidates for age transforms
MONTH_COLS = ["MoSold"]                                            # cyclical; or treat nominal

for df in (X_train, X_test):
    df["HouseAge"] = (df["YrSold"] - df["YearBuilt"]).clip(lower=0)
    df["YearsSinceRemodel"] = (df["YrSold"] - df["YearRemodAdd"]).clip(lower=0)
    # no-garage rows had GarageYrBlt filled with 0 -> use YearBuilt so the age is sane;
    # the HasGarage flag still carries the "no garage" information
    garage_year = df["GarageYrBlt"].where(df["GarageYrBlt"] > 0, df["YearBuilt"])
    df["GarageAge"] = (df["YrSold"] - garage_year).clip(lower=0)

    # binary indicators
    df["IsRemodeled"] = (df["YearRemodAdd"] != df["YearBuilt"]).astype(int)
    df["IsNew"]       = (df["YearBuilt"] == df["YrSold"]).astype(int)

    # single continuous time index (0-58) capturing price drift across the 2006-2010 window
    df["Month_since_start"] = (df["YrSold"] - 2006) * 12 + (df["MoSold"] - 1)

    df.drop(columns=YEAR_COLS + MONTH_COLS, inplace=True)


In [ ]:
NUMERIC_COLS = [
    "LotFrontage", "LotArea", "OverallQual", "OverallCond", "MasVnrArea",
    "BsmtFinSF1", "BsmtFinSF2", "BsmtUnfSF", "TotalBsmtSF",
    "1stFlrSF", "2ndFlrSF", "LowQualFinSF", "GrLivArea",
    "BsmtFullBath", "BsmtHalfBath", "FullBath", "HalfBath",
    "BedroomAbvGr", "KitchenAbvGr", "TotRmsAbvGrd", "Fireplaces",
    "GarageCars", "GarageArea",
    "WoodDeckSF", "OpenPorchSF", "EnclosedPorch", "3SsnPorch", "ScreenPorch",
    "PoolArea", "MiscVal",
    "HouseAge", "YearsSinceRemodel", "GarageAge", "Month_since_start",
    "TotalSF", "TotalBath", "TotalPorchSF", "OverallScore"
]

In [ ]:
DROP_COLS = ["Id", "Utilities"]   # Id: row identifier; Utilities: "AllPub" for all but 1 row

test_ids = X_test["Id"]   # keep for the submission file before dropping

X_train.drop(columns=DROP_COLS, inplace=True)
X_test.drop(columns=DROP_COLS, inplace=True)


Model testing

In [ ]:
print(make_mi_scores(X_train, y_train))

In [ ]:
X = X_train.copy()
y = y_train.copy()

cv = KFold(n_splits=5, shuffle=True, random_state=42)

xgb_param_dist = {
    'n_estimators': [100, 150, 200, 300, 400],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.1],
    'subsample': [0.6, 0.8, 1.0],
    'colsample_bytree': [0.6, 0.8, 1.0],
    'reg_alpha': [0, 0.1, 0.5, 1, 5],
    'reg_lambda': [0.5, 1, 2, 5, 10],
}

xgb_search = RandomizedSearchCV(
    XGBRegressor(random_state=42, eval_metric='logloss', tree_method='hist'),
    param_distributions=xgb_param_dist,
    n_iter=60,
    cv=cv,
    scoring='neg_root_mean_squared_error',
    random_state=42,
    n_jobs=-1,
)
xgb_search.fit(X, y)
best_xgb = xgb_search.best_estimator_

print(f"Best XGBoost CV Accuracy: {xgb_search.best_score_:.4f}")
print(f"Best XGBoost params: {xgb_search.best_params_}")

In [ ]:
test_preds_log = best_xgb.predict(X_test)          # predictions are on the log1p scale
test_preds = np.expm1(test_preds_log)              # undo the log transform -> dollars

submission = pd.DataFrame({"Id": test_ids, "SalePrice": test_preds})
submission.to_csv("submission.csv", index=False)   # gitignored by submission*.csv rule

print(submission.head())
print(submission["SalePrice"].describe())          # sanity-check: ~35k-750k, no negatives/NaN